In [1]:
%pip install -U langchain langchain-community langchain-huggingface langchain-google-genai pypdf faiss-cpu sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import  FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import RetrievalQA
from langchain.text_splitter import RecursiveCharacterTextSplitter

ModuleNotFoundError: No module named 'langchain.chains'

In [ ]:
file_path = "/Users/aaple/Documents/resume_analyzer/YousufBhatti_resume.pdf"
try:
    loader = PyPDFLoader(file_path=file_path)
    data = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        separators = ["\n\n", "\n", " ", ""]
    )
    chunks = text_splitter.split_documents(data)
except Exception as e:
    print(e)

In [ ]:
embeddings = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")

#### storing the embedings and chunks into FAISS

In [ ]:
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0, api_key=api_key)

In [ ]:
job_description = """Job Title: AI Software Engineer
Location: Remote / Hybrid
Job Type: Full-Time

Job Description:
We are looking for a passionate AI Software Engineer to design, develop, and deploy machine learning solutions.

Responsibilities:
- Implement and optimize ML models
- Develop AI features using Python, TensorFlow, PyTorch
- Work with large datasets and data pipelines
- Integrate AI models into production systems

Required Skills:
- Python, ML libraries (TensorFlow, PyTorch, scikit-learn)
- Data preprocessing and ML algorithms
- SQL/NoSQL databases
- Cloud platforms (AWS, GCP, Azure)

Preferred Skills:
- NLP / Computer Vision projects
- MLOps / model deployment pipelines
- Docker/Kubernetes
"""


In [ ]:
template = """
    You are an expert HR Recruiter. Compare the provided Resume Context with the Job Description.
    
    Resume Context: {context}
    Job Description: {question}
    
    Provide the output in this format:
    1. Match Percentage: (0-100%)
    2. Missing Keywords/Skills: 
    3. Experience Gap:
    4. Verdict: (Shortlist or Reject)
    
    Answer:"""

In [ ]:
from langchain.prompts import PromptTemplate
QA_PROMPT = PromptTemplate(template=template, input_variables=["context", "question"])

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),
        chain_type_kwargs={"prompt": QA_PROMPT}
    )


result = qa_chain.invoke({"query": job_description})


<class 'dict'>


In [ ]:
print(result['result'])

1.  **Match Percentage:** 50%
2.  **Missing Keywords/Skills:**
    *   TensorFlow
    *   PyTorch
    *   NoSQL (explicitly)
    *   Cloud platforms (AWS, GCP, Azure)
    *   Large datasets (in an engineering context)
    *   Data pipelines
    *   Integrate AI models into production systems
    *   MLOps / model deployment pipelines
    *   Docker/Kubernetes
    *   Optimize ML models (beyond basic implementation)
3.  **Experience Gap:** The candidate is a third-year Computer Science student seeking an entry-level role or internship, with strong foundational knowledge and academic projects. The job description is for a "Full-Time AI Software Engineer" which typically requires more hands-on experience in deploying, optimizing, and integrating AI models into production systems, working with large datasets, and utilizing specific industry tools like cloud platforms, MLOps, Docker/Kubernetes, and deep learning frameworks (TensorFlow/PyTorch). The candidate's experience is more aligned wit